--------------------------
# **NFL Temporal Language Shift Analysis**
--------------------------

**This research project will delve into the shift in language across 5 decades of NFL game commentary. The lens we are looking through to evaluate this revolves around player positions. In conjuction with vocabulary changes spanning from the 1970s to 2010s, we will address whether this shift impairs the ability of supervised classifiers to predict player position from commentary when trained and tested on different decades.**

--------------------------
## **Pipeline Breakdown**
--------------------------
**1. Data Loading and generating Token Blacklist**

**2. Pre-processing / Data Cleaning**

**3. Bootstrap log-ratio shift analysis with 95% Confidence Intervals**

**4. Rolling 5-year window Lexical divergence**

**5. Position Classification**

**6. Cross-decade transfer experiments**

**7. Significance Test**

--------------------------
## **Imported Libraries**

In [24]:
import json, re, random, warnings, os
import numpy as np
import pandas as pd
from collections import Counter
from itertools import combinations
import copy
 
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_extraction import text as sk_text
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from scipy.stats import chi2_contingency
from scipy.stats import chi2 as chi2_dist

# Setting a random seed for replicable results

random.seed(42)
np.random.seed(42)

--------------------------
## **Configuring**

This part outlines the pathing for the following 3 questions:

- Where is the data? 
- What files do I want to analyse
- Where do I want to save these results?

Additionally, I have restricted the analysis to the 1970s onwards due to extreme sparsity in the 1960s.

Limited the analysis to look at the 5 most notably positions, a mix of both offensive and defensive positions.

In [ ]:
#--------------------------------------------------------------------------------
# CONFIGURING THE DATA PATHING
#--------------------------------------------------------------------------------

# Have this ".ipynb" in the same folder as the dataset
## It will output results in a range of "csv" files
## Copies of the results have also been added to the "Appendices" of the pdf

# For the "DATA_DIR" adjust for where the data folder is stored

# The easiest way is to copy and paste "Copy Path" when right clicking the folder

#--------------------------------------------------------------------------------

DATA_DIR = r"C:\Users\thery\Downloads\dataset"  

MAIN_FILE = os.path.join(DATA_DIR, "FOOTBALL", "football_12.json")
ROSTER_FILE = os.path.join(DATA_DIR, "rosters", "NFL_team_rosters.json")
OUTPUT_DIR = "NFL_Temporal_Outputs"

POSITIONS = {"QB", "RB", "WR", "DB", "LB"}
MIN_YEAR = 1970

# Bootstrap Iterations for Confidence Intervals

N_BOOTSTRAP = 100

# Top Shifted Terms

TOPN = 25

os.makedirs(OUTPUT_DIR, exist_ok=True)

--------------------------
## **Loading the Data**

Above, the pathing for the data was outlined. 

Below we are doing the following:

- Loading the main dataset (commentary +labels)
- Loading the supporting metadata (NFL Team Rosters)
- Extracts a list of valid NFL Teams

In [5]:
#--------------------------------------------------------------------------------
# LOADING THE DATA
#--------------------------------------------------------------------------------

# Reads the JSON files and converts into a python object

# Only looking at NFL, not including NCAA (College/University Level)

#--------------------------------------------------------------------------------

def load_data():
    with open(MAIN_FILE, encoding = "utf-8") as f:
        data = json.load(f)

    with open(ROSTER_FILE, encoding = "utf-8") as f:
        nfl_rosters = json.load(f)

    return data, nfl_rosters

--------------------------
## **Building Token Blacklist**

In initial iterations, all named entities we removed. After consideration, some players may have names that overlap with common words, resulting in a more nuanced approach for named entity removal.

- Only block tokens that are *rare in English* and only appear in player names or team names
- Examples for names that are kept: "hill", "young", "long", "brown"
- Collect all named entity tokens, then remove any that also appear in NLTK word list
- Includes cleaning all names to a consistent format, for example: "Tom Brady" -> ["tom", "brady"] or "Mel Long" -> ["mel","long"]

In [6]:
#--------------------------------------------------------------------------------
# BUILDING TOKEN BLACKLIST
#--------------------------------------------------------------------------------

# 1. Build Common English word set

# 1.a) Include colours, physical descripters, adjectives / nouns and "football" terms

# 2. Build a function for obtaining tokens that are unique

# 2.a) Collects tokens found in player names then team names

# 2.b) Keeps only non-common words

# 2.c) Removes artefacts, exmaple "<player>" may have left behind "player"

#--------------------------------------------------------------------------------

COMMON_ENGLISH = set(sk_text.ENGLISH_STOP_WORDS) | {

    "brown", "white", "green", "gray", "grey", "black", "blue", "gold",
    
    "young", "long", "short", "little", "strong", "quick", "sharp",
    "hill", "wood", "ford", "field", "stone", "brooks", "rivers",
    "cross", "bell", "hall", "wells", "banks", "price", "king",
    "knight", "hunter", "fisher", "cook", "marshall", "rice",
    
    "yard", "yards", "touchdown", "pass", "run", "kick", "return",
    "tackle", "block", "catch", "throw", "snap", "rush", "score",
    "line", "field", "end", "back", "side", "zone",
}

def build_blacklist(nfl_rosters):
    name_token_counts = Counter()
    for team, years in nfl_rosters.items():
        for _, players in years.items():
            for full_name in players:
                for tok in re.findall(r"[a-z]+", full_name.lower()):
                    if len(tok) > 2:
                        name_token_counts[tok] += 1
 
    team_tokens = set()
    for team in nfl_rosters:
        for tok in re.findall(r"[a-z]+", team.lower()):
            if len(tok) > 2:
                team_tokens.add(tok)
 
    blacklist = set()
    for tok, _ in name_token_counts.items():
        if tok not in COMMON_ENGLISH:
            blacklist.add(tok)
    for tok in team_tokens:
        if tok not in COMMON_ENGLISH:
            blacklist.add(tok)
 
    blacklist.update({"gt", "player", "lt"})

    return blacklist

#--------------------------------------------------------------------------------
# CLEANING FUNCTION
#--------------------------------------------------------------------------------

# Original Text example: 
## "Tom Brady throws to <player> near the sideline for New England"

# Cleaned Text example:
## "throws to near the sideline for"

#--------------------------------------------------------------------------------
 
def clean_text(text: str, blacklist: set) -> str:
    text = re.sub(r"<[^>]+>", " ", text.lower())
    toks = re.findall(r"[a-z']+", text)
    toks = [t for t in toks if len(t) > 1 and t not in blacklist]
    return " ".join(toks)
 

--------------------------
## **Building the Dataframe**

When building the dataframe we want to accomplish the following 3 tasks, to prepare for analysis:

1. Filters out the data
2. Build the text field in a consistent manner
3. Establish a structured pandas dataframe

In [ ]:
#--------------------------------------------------------------------------------
# BUILDING THE DATAFRAME
#--------------------------------------------------------------------------------

# Extract the NFL Team Names

# Loop through the data and obtain label information: "teams", "position", "year"

# Filter out non-NFL

# Filter by position and year (set in Configuration step)

# Build raw text then puts in structured rows and cleans

# Converts to a oandas dataframe

#--------------------------------------------------------------------------------

def build_df(data, nfl_rosters, blacklist):
    nfl_teams = set(nfl_rosters.keys())
    rows = []

    for _, item in data.items():
        label = item["label"]
        teams = label["teams"]

        if not all(t in nfl_teams for t in teams):
            continue

        pos = label["position"]
        year = int(label["year"])

        if pos not in POSITIONS or year < MIN_YEAR:
            continue

        raw_text = " ".join(
            tok for tok in item["mention"]
            if tok and tok != "<player>"
        )

        rows.append({
            "year"    : year,
            "decade"  : (year // 10) * 10,
            "position": pos,
            "text"    : clean_text(raw_text, blacklist),
        })
        
    df = pd.DataFrame(rows)

    print(f"Total rows after filtering: {len(df)}")
    print(df.groupby(["decade", "position"]).size().unstack(fill_value=0).to_string())
    
    return df


--------------------------
## **Implementation of Custom Stop Words**

Build a set of tailored stop words for the dataset.

In [9]:
#--------------------------------------------------------------------------------
# BUILDING A SET OF STOP WORDS
#--------------------------------------------------------------------------------

# Creates a set of the union of "classic" stop words and custom additions

# Includes addition words that will be ignored during feature extraction

#--------------------------------------------------------------------------------

CUSTOM_STOP = {
    "just", "good", "got", "going", "game", "play", "time",
    "yard", "yards", "line", "ball", "second", "right", "left",
    "first", "one", "two", "three", "four", "five", "six",
    "seven", "eight", "nine", "ten", "back", "down", "really",
    "little", "lot", "look", "looking", "come", "comes", "say",
    "said", "player",
}

ANALYSIS_STOP = set(sk_text.ENGLISH_STOP_WORDS) | CUSTOM_STOP

---------------------------
# **Temporal Analysis Overview**
---------------------------
## **Finds which words change over time: Bootstrap Shift**

Why bootstrap over other techniques? 

Instead of just computing one value:
- Resample the data many times
- Computes the distributions of shifts
- Gets CIs
- Can prevent "false positives" caused by smaller sample sizes in the 1970s data

## **Measuring how much language has changed: Rolling Lexical Divergence**

We could have just look at 1970s vs 2010s. To gain more insight we can implement this approach by:
- Splitting the data into 5-year segments (not just looking at decades as a whole but in decades too)
- Building TF-IDF vectors for each respective vector
- Cosine similarity in order to compare

## **Testing whether language change affects NLP Performance: Classification**

How can we assess if there is a significant shift in language?

The Rolling Lexical Divergence gives a measure of change, whereas our classification delves into the change of language paired with the impact on NLP models. A variety of approachs are used to address this:
1. Within-Decade: Train and Test on the same decade (Theoretical Best-Case Performace)
2. Cross-Decade: Train and Test on different decades (Generalisation across Time)
3. Multiple Models: Naives Bayes, Logisitic Regression, Linear SVM (Verify results aren't model specific)
4. Cross-Validation: Stratified CV (More reliable as it isn't a random split)
5. Majority-Class Basline: Predict the most common class
6. McNemar Test: Provides the "p-value" needed to claim language shift is significant

--------------------------

## **Bootstrap Shift**

To get our answer:
1. Estimates word probabilities in each era
2. Taking a log-ratio to measure change
3. Use resampling to get 95% CIs
4. Keeping only statistically reliable shifts

In [36]:
#--------------------------------------------------------------------------------
# BOOTSTRAP SHIFT
#--------------------------------------------------------------------------------

# Set out our inputs
## List of docs from corpus'
## N_BOOTSTRAP is initialised in the Configuration
## TOPN is initialised in the Configuration
## Min num of docs a term has to appear in
## Max num of texts to use from each corpus 
## Limits set for practicality, so runtime isn't too excessive

# Set identical random seed for consistent replicable results

# Vectorizer inputs
## Ignores custom stop words
## min_df provides minimum num of occurences to remove rare words
## Keep unigrams and bigrams
## max_features limits the vocabulary size

# Fits to an array

#--------------------------------------------------------------------------------

def bootstrap_shift(texts_a: list, texts_b: list,
                    n_bootstrap: int = N_BOOTSTRAP,
                    topn: int = TOPN,
                    min_df: int = 30,
                    max_sample: int = 5000):
        
    rng = np.random.default_rng(42)

    # If there are more texts, than predetermined maximum
    ## Take "max_sample" number of randomly chosen texts 
    ## Sample without replacement

    if len(texts_a) > max_sample:
        texts_a = [texts_a[i] for i in rng.choice(len(texts_a), max_sample, replace=False)]

    if len(texts_b) > max_sample:
        texts_b = [texts_b[i] for i in rng.choice(len(texts_b), max_sample, replace=False)]

    # Concatenates to create one combined corpus
    ## Means both corpa must be compared using the same feature space

    all_texts = texts_a + texts_b

    # Converts text into doc-term matrix

    vec = CountVectorizer(
        stop_words = list(ANALYSIS_STOP),
        min_df = min_df,
        ngram_range = (1, 2),
        max_features = 15000,
    )
    
    # Vectorizer learns the vocab from all the texts
    ## Stores in an array

    vec.fit(all_texts)
    terms = np.array(vec.get_feature_names_out())

    #--------------------------------------------------------------------------------
    # EXTRACTING TERM PROBABILITIES
    #--------------------------------------------------------------------------------

    # Converts Corpus to Matrix

    # Adds up term count across all docs, flatterns to 1D
    ## The +1 removes chance of trying to divide by 0

    # Returns the relative frequencies of terms

    # Computes log-ratio for each term
    ## pa and pb denote term probabilities for each corpus

    #--------------------------------------------------------------------------------

    def corpus_term_prob(texts):

        X = vec.transform(texts)
        counts = np.asarray(X.sum(axis=0)).ravel().astype(float) + 1

        return counts / counts.sum()
    
    pa = corpus_term_prob(texts_a)
    pb = corpus_term_prob(texts_b)

    obs_shift = np.log(pb / pa)

    #--------------------------------------------------------------------------------
    # RESAMPLING EACH CORPUS
    #--------------------------------------------------------------------------------

    # Create zero matrix to store bootstrap results: Bootstrap Iterations x Terms

    # Perform our Bootstrap Loop
    ## Samples with replacement: docs may appear multiple times and others not at all
    ## Estimates the term distributions for this specific sample
    ## Add to our zero matrix 

    # Results in a complete sampling distribution to obtain our 95% CI

    #--------------------------------------------------------------------------------

    boot_shifts = np.zeros((n_bootstrap, len(terms)))
    n_a, n_b = len(texts_a), len(texts_b)

    for i in range(n_bootstrap):

        sample_a = [texts_a[j] for j in np.random.randint(0, n_a, n_a)]
        sample_b = [texts_b[j] for j in np.random.randint(0, n_b, n_b)]

        fa = corpus_term_prob(sample_a)
        fb = corpus_term_prob(sample_b)

        boot_shifts[i] = np.log(fb / fa)
 
    ci_low = np.percentile(boot_shifts, 2.5,  axis=0)
    ci_high = np.percentile(boot_shifts, 97.5, axis=0)


    #--------------------------------------------------------------------------------
    # DETERMINING SIGNIFICANCE
    #--------------------------------------------------------------------------------

    # A term is significant if the whole CI is either above or below 0
    ## If it doesn't cross 0, then it suggests the direction of change is stable

    # Builds a visualisation of results in a pandas dataframe
    ## Outputs:
    ## The word or bigram
    ## Observed shift
    ## Confidence Interval Bounds
    ## Is the shift significant

    # Outputs the significant terms  

    #--------------------------------------------------------------------------------

    term_significance = (ci_low > 0) | (ci_high < 0)

    shift_df = pd.DataFrame({
        "term" : terms,
        "log_ratio" : obs_shift,
        "Confidence_Interval_Low" : ci_low,
        "Confidence_Interval_High" : ci_high,
        "Significance" : term_significance,
    })

    increasing_terms = shift_df[shift_df["Significance"]].nlargest(topn, "log_ratio")
    decreasing_terms = shift_df[shift_df["Significance"]].nsmallest(topn, "log_ratio")

    return increasing_terms, decreasing_terms

--------------------------

## **Rolling Lexical Divergence**

As mentioned earlier, it would be more trivial to just compare the 1970s to the 2010s. In this analysis we compare 5-year segments against other segments to understand if and where gradual evolution occurs, for both years and specific player positions.

In [42]:
#--------------------------------------------------------------------------------
# ROLLING LEXICAL DIVERGENCE
#--------------------------------------------------------------------------------

# Set out our inputs
## Pandas Dataframe (Will make a copy to avoid modifying the original)
## Set our 5-year segments for a more in-depth analysis

# Creates a copy
## Sorts the years into 5-year segments 
## e.g. 2014 -> 2010-2014 segment and 1986 -> 1985 - 1989 segment

#--------------------------------------------------------------------------------

def rolling_divergence(df: pd.DataFrame, segment: int = 5):

    df = df.copy()
    df["segment"] = (df["year"] // segment) * segment
    rows = []

    #--------------------------------------------------------------------------------
    # LOOP FOR EACH POSITION
    #--------------------------------------------------------------------------------

    # Looks at each pre-selected position seperately e.g. QB or LB
    ## Language may vary based on this

    # Merges text for each 5-year period into one doc

    # Skips if there isn't enough data

    # Buils TF-IDF representation for these 5 year bins

    # Calculates Cosine Similarity
    ## Get a ordered list of each of the 5-year segments

    #--------------------------------------------------------------------------------

    for pos in sorted(df["position"].unique()):

        specific_pos = df[df["position"] == pos]
        group_text = specific_pos.groupby("segment")["text"].apply(lambda s: " ".join(s))

        if len(group_text) < 2:
            continue

        segment_tfidf = TfidfVectorizer(stop_words = "english", min_df = 2, max_features = 5000)

        try:
            X = segment_tfidf.fit_transform(group_text.values)
        
        except ValueError:
            continue

        cosine_score = cosine_similarity(X)
        segments = list(group_text.index)

        #--------------------------------------------------------------------------------
        # COMPARING EACH SEGMENT AND PRODUCING RESULTS TABLE
        #--------------------------------------------------------------------------------

        # Looks at every pair e.g. (1970, 1975), (1970, 1980), ...
        ## Stores the results for the comparison in a pandas table, with these fields:
        ## Player Position
        ## Early Time Window
        ## Late Time Window
        ## Time Difference between them
        ## Similarity between them
        ## Distance between them (1 - Cosine Similarity)

        # Returns the results as a pandas dataframe

        #--------------------------------------------------------------------------------

        for i, j in combinations(range(len(segments)), 2):
            rows.append({
                "Positions" : pos,
                "Early_Time_Window" : int(segments[i]),
                "Late_Time_Window" : int(segments[j]),
                "Time_Gap" : int(segments[j] - segments[i]),
                "Cosine_Similarity" : float(cosine_score[i, j]),
                "Distance" : float(1 - cosine_score[i, j]),
            })

    return pd.DataFrame(rows)


--------------------------

## **Classification**

We can split this part into 4 distinct sections that encapsulates the initially outlined approaches to answer "Does Temporal Drift hurt Classification performance?"

1. Pipelines (Chosen Models and Features)
2. Within-Decade Evaluation (Theoretical Best-Case)
3. Cross-Decade Evaluation (Generalisation across Time)
4. Baselines and Significance (Are these differences real?)

--------------------------

## **1. Pipelines**

Our setup for this evaluation consistents of sparse TF-IDF Vectors along with multi-class classification (QB, RB, WR, DB, LB). So implementing models effective for this is important. We are looking at 3 models to establish that findings aren't model specific.

1. Naive Bayes: Classic Baseline for text classification as it assumes words are generated independently given class
2. Logistic Regression: Learns weightings, so each word contributes differently to each class
3. Linear SVM: Instead of probabilities it finds planes that best seperates the classes



In [53]:
#--------------------------------------------------------------------------------
# MAKING PIPELINES FOR EACH MODEL
#--------------------------------------------------------------------------------

# Set out our models in the form of a dictionary that contains each
## Contains both TF-IDF Vectorisation and the chosen classifier 

# Using "Pipeline()" keeps pre-processing and modelling together
## Useful for CV later on

#--------------------------------------------------------------------------------

def models():
    return {
        "NaiveBayes": Pipeline([
            ("tfidf", TfidfVectorizer(
                stop_words = "english", ngram_range = (1, 2),
                min_df = 2, max_features = 5000)),
            ("clf", MultinomialNB()),
        ]),
        "LogisticRegression": Pipeline([
            ("tfidf", TfidfVectorizer(
                stop_words = "english", ngram_range = (1, 2),
                min_df = 2, max_features = 5000)),
            ("clf", LogisticRegression(
                max_iter = 1000, C = 1.0, solver = "lbfgs", random_state = 42)),
        ]),
        "LinearSVC": Pipeline([
            ("tfidf", TfidfVectorizer(
                stop_words = "english", ngram_range = (1, 2),
                min_df = 2, max_features = 5000)),
            ("clf", LinearSVC(max_iter = 2000, C = 1.0, random_state = 42)),
        ]),
    }

#--------------------------------------------------------------------------------
# SETTING UP A BALANCED SAMPLE DATASET
#--------------------------------------------------------------------------------

# Initial results were skewed due to some classes having more data than others

# Now we can set up a sample of (for example) 300 of each class for fairness

#--------------------------------------------------------------------------------

def balanced_dataset(df, positions, n_class, seed = 42):

    rows = []

    #--------------------------------------------------------------------------------
    # LOOP FOR EACH POSITION
    #--------------------------------------------------------------------------------

    # Looks at each pre-selected position seperately e.g. QB or LB
    ## Filters for the respective position

    # Checks there are enough players for that position

    # Samples the same number from each class, with constant random seed

    # Joins all the samples together, shuffles the rows and resets the indexing

    #--------------------------------------------------------------------------------

    for pos in positions:

        filtered_pos = df[df["position"] == pos]

        if len(filtered_pos) < n_class:
            return None
        
        rows.append(filtered_pos.sample(n_class, random_state = 42))
    
    join_shuffle_class = pd.concat(rows).sample(frac = 1, random_state = seed). reset_index(drop = True)

    return join_shuffle_class

--------------------------

## **2. With-In Decade Evaluation**

This look at our "Best Case" evaluation, where we train and test classification on the same decade.

1. Set up a Stratified fold
2. Choose the specific decade
3. Use the previously set balanced position dataset to create a "fair" split
4. Set a loop to obtain cross-validation of 5-folds for each model
5. Store results in a pandas dataframe, including computed metrics

In [46]:
#--------------------------------------------------------------------------------
# WITHIN-DECADE
#--------------------------------------------------------------------------------

# 5-Fold Cross-Validation to obtain "reliable" performance
## Utilise stratified for strong split
## Splits data into 5 equal folds
## Train on 4 folds and Test on 1 fold

# Use the balanced out dataset
## Filters to one decade
## Skips if the decade doesn't contain enough data

#--------------------------------------------------------------------------------

def within_decade_cv(df, focus_decades, focus_pos, n_class = 300):

    rows = []
    skf = StratifiedKFold(n_splits = 5, shuffle = True, random_state = 42)

    for dec in focus_decades:

        split = balanced_dataset(df[df["decade"] == dec], focus_pos, n_class)

        if split is None:
            print(f"Skipping Decade {dec}: Insufficient Data")
            continue

    #--------------------------------------------------------------------------------
    # LOOP FOR EACH CLASSIFIER
    #--------------------------------------------------------------------------------

    # Takes each model from the initialised "models" function

    # Runs a cross-validation predicttion on the shuffled split generated

    # Stores the results that get returned in a pandas sataframe
    ## Decade
    ## Classification model
    ## Computes and stores metrics: Accuracy and Average F1 for each class

    #--------------------------------------------------------------------------------

        for clf_name, pipe in models().items():

            preds = cross_val_predict(pipe, split["text"], split["position"], cv = skf)

            rows.append({
                "Decade" : dec,
                "Classifier" : clf_name,
                "Accuracy" : accuracy_score(split["position"], preds),
                "Macro_F1" : f1_score(split["position"], preds, average = "macro"),
            })

    return pd.DataFrame(rows)


--------------------------

## **3. Cross-Decade Evaluation**

This analysis employs a similar approach to "With-In Decade" witht he added layer of training on a decade and testing on a different one, whereas the prior section trains and tests for the same decade. These parts independently may not tell us too much, however together we can compare to assess if language drift has effected classification. 

If this has occured, we would expect to see:
- With-In Decade: Stronger Results
- Cross- Decade: Weaker Results

In [50]:
#--------------------------------------------------------------------------------
# CROSS-DECADE
#--------------------------------------------------------------------------------

# Obtains a training set for Decade "x"

# Uses the balancing function for "n" random points

#--------------------------------------------------------------------------------

def cross_decade(df, focus_decades, focus_pos, n_class = 300):

    rows = []
    model_use = models()

    for train_dec in focus_decades:

        train_split = balanced_dataset(df[df["decade"] == train_dec], focus_pos, n_class)

        if train_split is None:
            continue

    #--------------------------------------------------------------------------------
    # LOOP FOR TESTING DECADES
    #--------------------------------------------------------------------------------

    # Obtains a testing set for Decade "y"

    # Uses the balancing function for "n" random points

    #--------------------------------------------------------------------------------

        for test_dec in focus_decades:

            test_split = balanced_dataset(df[df["decade"] == test_dec], focus_pos, n_class)

            if test_split is None:
                continue

            #--------------------------------------------------------------------------------
            # LOOP FOR EACH CLASSIFIER
            #--------------------------------------------------------------------------------

            # Takes each model from the initialised "models" function

            # Fits the model using both TF-IDF and Classifier

            # Predicts on test set

            # Stores the results that get returned in a pandas sataframe
            ## Training Decade
            ## Testing Decade
            ## Classification model
            ## Computes and stores metrics: Accuracy and Average F1 for each class

            #--------------------------------------------------------------------------------

            for clf_name, mod in model_use.items():

                mod.fit(train_split["text"], train_split["position"])
                preds = mod.predict(test_split["text"])

                rows.append({
                    "Training_Decade" : train_dec,
                    "Testing_Decade" : test_dec,
                    "Classifier" : clf_name,
                    "Accuracy" : accuracy_score(test_split["position"], preds),
                    "Macro_F1" : f1_score(test_split["position"], preds, average = "macro"),
                })

    return pd.DataFrame(rows)


--------------------------

## **4. Baselines and Significance**

For this section we are performing a test for determining the "baseline" which will tell us if the models are doing anything effective. This isn't expected to be a strong test, so it should be outclassed by the previous tests that we've set up. 

Additionally, we are performing a McNemar Significance Test. This will reveal whether the "theoretical" drop in performance between "With-In Decade" and "Cross-Decade" are statistically significant. We can do this by training 2 models, the first is trained and tested on the 2010s data and the second is trained on the 1970s and tested on the same 2010s test set. For this we are applying the Continuinty-Corrected McNemar Chi-Square Statistic: $\frac{(|b-c|-1)^2}{b+c}$



In [ ]:
#--------------------------------------------------------------------------------
# BASELINE
#--------------------------------------------------------------------------------

# This will predict the most frequent class, which should be 0.2
## We use 5 classes and the balancing function makes sure there are equal volumes

# Looks at a specific Decade and extracts a sample

# Finds the modal class

# Predicts the class for each row (should be 0.2)

#--------------------------------------------------------------------------------

def baseline(df, focus_decades, focus_pos, n_class = 300):

    rows = []

    for dec in focus_decades:
        split = balanced_dataset(df[df["decade"] == dec], focus_pos, n_class)

        if split is None:
            continue

        base = split["position"].mode()[0]

        preds = [base] * len(split)

        rows.append({
            "Decade"    : dec,
            "Classifier": "Baseline",
            "Accuracy"  : accuracy_score(split["position"], preds),
            "Macro_F1"  : f1_score(split["position"], preds, average = "macro", zero_division = 0),
        })
    return pd.DataFrame(rows)

#--------------------------------------------------------------------------------
# MCNEMAR SIGNIFICANCE TEST
#--------------------------------------------------------------------------------

# We are using 150 opposed to 300 as a base
## As we require multiple independent datasets
## We ensure enough data for each class and a more manageable runtime

# Using a different but constant seed to have fresh splits from previous parts

# Create our training sets and test set

#--------------------------------------------------------------------------------

def mcnemar_signif(df, focus_pos, n_class = 150):

    rows = []
    dec_a, dec_b = 1970, 2010
 
    train_a = balanced_dataset(df[df["decade"] == dec_a], focus_pos, n_class, seed = 24)
    train_b = balanced_dataset(df[df["decade"] == dec_b], focus_pos, n_class, seed = 24)

    test_b  = balanced_dataset(df[df["decade"] == dec_b], focus_pos, n_class, seed = 17)
 
    if any(x is None for x in [train_a, train_b, test_b]):
        print("Insufficient data for McNemar test")
        return pd.DataFrame()
    
    #--------------------------------------------------------------------------------
    # TRAINING MODELS
    #--------------------------------------------------------------------------------

    # With-In Decade: Train -> 2010s, Test -> 2010s
    
    # Cross- Decade: TRain -> 1970s, Test -> 2010s

    # Gives us a boolean array for correct or not predictions

    #--------------------------------------------------------------------------------
 
    for clf_name, mod in models().items():
        
        mod_within = copy.deepcopy(mod)
        mod_within.fit(train_b["text"], train_b["position"])

        pred_within = mod_within.predict(test_b["text"])
        correct_within = (pred_within == test_b["position"].values)
 

        mod_cross = copy.deepcopy(mod)
        mod_cross.fit(train_a["text"], train_a["position"])

        pred_cross = mod_cross.predict(test_b["text"])
        correct_cross = (pred_cross == test_b["position"].values)

        #--------------------------------------------------------------------------------
        # PERFORMING MCNEMAR TEST
        #--------------------------------------------------------------------------------

        # McNemar's test is a non-parametric statistical test

        # b: With-In Decade model is correct and Cross-Decade model is wrong

        # c: With-In Decade model is wrong and Cross-Decade model is correct

        # Apply the test statistic

        # Put values into a pandas dataframe
        ## Classifier USed
        ## b
        ## c
        ## McNemar Chi-Squared value
        ## p-value obtained
        ## Is it significant?

        #--------------------------------------------------------------------------------
 
        b = int(np.sum( correct_within & ~correct_cross)) 
        c = int(np.sum(~correct_within &  correct_cross))  
 
        if (b + c) > 0:
            chi2_stat = (abs(b - c) - 1) ** 2 / (b + c)
            p_val = float(1 - chi2_dist.cdf(chi2_stat, df = 1))

        else:
            chi2_stat, p_val = 0.0, 1.0
 
        rows.append({
            "Classifier" : clf_name,
            "With_In_Correct": b,
            "Cross_Correct": c,
            "McNemar_Chi_Squared" : round(chi2_stat, 4),
            "p_value" : round(p_val, 6),
            "Significance" : p_val < 0.05,
        })

    return pd.DataFrame(rows)


--------------------------
# **Putting Everything All Together**
--------------------------

Above we have built all the necessary functions to try and answer our question:

*NFL Commentary Analysis: A Study of Temporal Language Drift in NFL Commentary Using Lexical Shift Analysis and Classification Models*

Below is the part that orchestrates the whole pipeline which we stablished at the start.

1. Loading the Data
2. Running our Token Blacklist on the Data
3. Creates the Dataframe
4. Performs our Language SHift Analysis for 1970s vs 2010s: Bootstrap
5. Performs our Player Position Shift Analysis: Bootstrap
6. Performs our Rolling Lexical Divergence
7. Performs our With-In Decade Evaluation
8. Performs our Cross-Decade Evaluation
9. Sets our Baseline
10. Performs our McNemar Significance Test

--------------------------

The results are stored in *csv* files, and the key results will be replicated in tables in the report. **Note that all results will be in these *csv* files.**

--------------------------

In [ ]:
#--------------------------------------------------------------------------------
# MAIN
#--------------------------------------------------------------------------------

def main():

    print("=" * 60)
    print("Loading data...")
    data, nfl_rosters = load_data()
 
    print("Building token blacklist...")
    blacklist = build_blacklist(nfl_rosters)
    print(f"  Blacklist size: {len(blacklist)} tokens")
 
    print("Building dataframe...")
    df = build_df(data, nfl_rosters, blacklist)
 
    counts = pd.crosstab(df["decade"], df["position"])
    counts.to_csv(f"{OUTPUT_DIR}/counts_by_decade_position.csv")
 
    #--------------------------------------------------------------------------------
    # BOOTSTRAP: SHIFT ANALYSIS
    #--------------------------------------------------------------------------------

    # Performing the analysis between 1970s and 2010s explicitly
    ## Setting the text from each chosen decade to a list
    ## Uses bootstrap to retrieve the "TOPN" increasing/decreasing terms per decade

    # Results are then placed into a csv file
    ## Outputs the top 5 of each

    #--------------------------------------------------------------------------------

    print("\nRunning bootstrap shift analysis (1970s vs 2010s)...")

    texts_1970 = df[df["decade"] == 1970]["text"].tolist()
    texts_2010 = df[df["decade"] == 2010]["text"].tolist()

    increasing_terms, decreasing_terms = bootstrap_shift(texts_1970, texts_2010, topn = TOPN)
 
    out = pd.concat([
        increasing_terms.rename(columns={"term": "term_increasing",  "log_ratio": "lr_increasing",
                                "Confidence_Interval_Low": "ci_lo_increase", "Confidence_Interval_High": "ci_hi_increase"}
                      ).reset_index(drop=True),
        decreasing_terms.rename(columns={"term": "term_decreasing", "log_ratio": "lr_decreasing",
                                 "Confidence_Interval_Low": "ci_lo_decrease", "Confidence_Interval_High": "ci_hi_decrease"}
                       ).reset_index(drop=True),
    ], axis=1)

    out.to_csv(f"{OUTPUT_DIR}/shift_1970s_vs_2010s_overall_bootstrapped.csv", index = False)

    print(f"  Top Increasing Terms (2010s): {increasing_terms['term'].head(5).tolist()}")
    print(f"  Top Decreasing Terms (1970s): {decreasing_terms['term'].head(5).tolist()}")
 
    #--------------------------------------------------------------------------------
    # BOOTSTRAP: PLAYER POSITION ANALYSIS
    #--------------------------------------------------------------------------------

    # Repeats the Shift Analysis for specific player positions

    #--------------------------------------------------------------------------------

    print("\nRunning position-specific shift analyses...")
    for pos in sorted(POSITIONS):
        t1970 = df[(df["decade"] == 1970) & (df["position"] == pos)]["text"].tolist()
        t2010 = df[(df["decade"] == 2010) & (df["position"] == pos)]["text"].tolist()

        if len(t1970) < 50 or len(t2010) < 50:
            print(f"  {pos}: insufficient data, skipping")
            continue

        increase, decrease = bootstrap_shift(t1970, t2010, topn = 15, min_df = 10)

        out = pd.concat([
            increase[["term","log_ratio","Confidence_Interval_Low","Confidence_Interval_High"]].rename(
                columns={"term":"more_2010s","log_ratio":"lr_up"}
            ).reset_index(drop=True),
            decrease[["term","log_ratio","Confidence_Interval_Low","Confidence_Interval_High"]].rename(
                columns={"term":"more_1970s","log_ratio":"lr_down"}
            ).reset_index(drop=True),
        ], axis=1)

        out.to_csv(f"{OUTPUT_DIR}/shift_1970s_vs_2010s_{pos}_bootstrapped.csv", index = False)

        print(f"  {pos} done (1970s mentions: {len(t1970)}, 2010s mentions: {len(t2010)})")
 
    #--------------------------------------------------------------------------------
    # ROLLING LEXICAL DIVERGENCE
    #--------------------------------------------------------------------------------

    # Performs our rolling lexical divergence

    #--------------------------------------------------------------------------------

    print("\nComputing Rolling Lexical Divergence...")

    div_df = rolling_divergence(df, segment = 5)
    div_df.to_csv(f"{OUTPUT_DIR}/rolling_lexical_divergence.csv", index = False)

    div_decade = div_df[div_df["Time_Gap"] % 10 == 0].copy()
    div_decade.to_csv(f"{OUTPUT_DIR}/decade_lexical_divergence.csv", index = False)

    print(f"  {len(div_df)} Segment-Pair Distances")
 
    #--------------------------------------------------------------------------------
    # WITH-IN DECADE: CLASSIFICATION
    #--------------------------------------------------------------------------------

    # Choses specific decades and positions to assess to look at

    # Performs our evaluation

    #--------------------------------------------------------------------------------

    focus_decades = [1970, 1990, 2010]
    focus_pos = ["QB", "RB", "WR", "DB", "LB"]
 
    print("\nRunning With-In Decade Classification...")

    within_df = within_decade_cv(df, focus_decades, focus_pos)
    within_df.to_csv(f"{OUTPUT_DIR}/within_decade_cv_results.csv", index = False)

    print(within_df.to_string(index = False))
 
    #--------------------------------------------------------------------------------
    # CROSS-DECADE: CLASSIFICATION
    #--------------------------------------------------------------------------------

    # USes the same focus points as With-In

    # Builds a matrix with training vs test decades and showing Macro F1 Score

    #--------------------------------------------------------------------------------

    print("\nRunning Cross-Decade...")

    cross_df = cross_decade(df, focus_decades, focus_pos)
    cross_df.to_csv(f"{OUTPUT_DIR}/cross_decade_transfer_results.csv", index = False)
 
    lr_pivot = (
        cross_df[cross_df["Classifier"] == "LogisticRegression"]
        .set_index(["Training_Decade", "Testing_Decade"])["Macro_F1"]
        .unstack()
    )

    print("\nLogistic Regression macro-F1 (train→test):")
    print(lr_pivot.to_string())
 
    #--------------------------------------------------------------------------------
    # BASLINE: CLASSIFICATION
    #--------------------------------------------------------------------------------

    # Stores the Baseline results in a csv file
    #--------------------------------------------------------------------------------

    print("\nComputing majority-class baseline...")

    base_df = baseline(df, focus_decades, focus_pos)
    base_df.to_csv(f"{OUTPUT_DIR}/baseline_results.csv", index = False)

    print(base_df.to_string(index = False))
 
    #--------------------------------------------------------------------------------
    # MCNEMAR SIGNIFICANCE TEST: CLASSIFICATION
    #--------------------------------------------------------------------------------

    # Performs our test and stores results in a csv file

    #--------------------------------------------------------------------------------

    print("\nRunning McNemar significance tests (1970s vs 2010s)...")

    mcn_df = mcnemar_signif(df, focus_pos)

    if not mcn_df.empty:
        mcn_df.to_csv(f"{OUTPUT_DIR}/mcnemar_results.csv", index = False)
        print(mcn_df.to_string(index = False))
 
    print(f"\n{'='*60}")
    print(f"All outputs saved to: {OUTPUT_DIR}")
    print(f"{'='*60}")
 
 
if __name__ == "__main__":
    main()


Loading data...
Building token blacklist...
  Blacklist size: 10954 tokens
Building dataframe...
Total rows after filtering: 167340
position     DB    LB     QB     RB     WR
decade                                    
1970       1901  1676   3160   3498   2531
1980       2345  1629   5814   3439   3419
1990       4116  2132   6008   3789   4467
2000       9285  3641  21167  11286  10581
2010      11537  5329  17707  12388  14495

Running bootstrap shift analysis (1970s vs 2010s)...
  Top Increasing Terms (2010s): ['route', 'huge', 'special', 'nice', 'guys']
  Top Decreasing Terms (1970s): ['replacing', 'number number', 'men', 'bench', 'pattern']

Running position-specific shift analyses...
  DB done (1901: 1970s mentions, 11537: 2010s mentions)
  LB done (1676: 1970s mentions, 5329: 2010s mentions)
  QB done (3160: 1970s mentions, 17707: 2010s mentions)
  RB done (3498: 1970s mentions, 12388: 2010s mentions)
  WR done (2531: 1970s mentions, 14495: 2010s mentions)

Computing Rolling Lex